<a href="https://colab.research.google.com/github/Rishii077/AI-Lab-Assignments/blob/main/Experiment_12_Mini_Project_Capstone.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q scikit-learn streamlit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 69.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 59.8 MB/s eta 0:00:00


In [2]:
import re
import math
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("Libraries imported successfully!")

Libraries imported successfully!


In [3]:
documents = [
    """
    Artificial Intelligence is a branch of computer science that focuses
    on creating systems capable of performing tasks that normally require
    human intelligence. AI is used in healthcare, education, finance,
    transportation and many other fields.
    """,

    """
    Machine Learning is a subset of Artificial Intelligence. It allows
    computers to learn patterns from data and make predictions or decisions
    without being explicitly programmed for every task.
    """,

    """
    Deep Learning is a subset of Machine Learning that uses artificial
    neural networks with multiple layers. It is commonly used for image
    recognition, speech recognition and natural language processing.
    """,

    """
    Retrieval-Augmented Generation, or RAG, combines information retrieval
    with generative AI. A RAG system retrieves relevant information from
    a knowledge source and uses that information to generate an answer.
    """,

    """
    AI in healthcare can assist with medical image analysis, disease risk
    prediction, clinical documentation and healthcare data analysis.
    AI systems should be carefully evaluated because incorrect predictions
    can affect medical decisions.
    """,

    """
    Natural Language Processing, or NLP, is a field of Artificial
    Intelligence that enables computers to understand and process human
    language. NLP is used in chatbots, translation, summarization and
    sentiment analysis.
    """
]

print("Knowledge base created successfully!")
print("Number of documents:", len(documents))

Knowledge base created successfully!
Number of documents: 6


In [4]:
vectorizer = TfidfVectorizer()

document_vectors = vectorizer.fit_transform(documents)

print("RAG indexing completed successfully!")
print("Document vectors shape:", document_vectors.shape)

RAG indexing completed successfully!
Document vectors shape: (6, 105)


In [5]:
def retrieve_documents(question, top_k=2):

    question_vector = vectorizer.transform([question])

    similarity_scores = cosine_similarity(
        question_vector,
        document_vectors
    )[0]

    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    retrieved_documents = []

    for index in top_indices:
        retrieved_documents.append({
            "document": documents[index],
            "score": similarity_scores[index]
        })

    return retrieved_documents

In [6]:
question = "What is Artificial Intelligence?"

retrieved = retrieve_documents(question)

print("========== RETRIEVED DOCUMENTS ==========")

for item in retrieved:
    print("\nSimilarity Score:", round(item["score"], 3))
    print(item["document"])

========== RETRIEVED DOCUMENTS ==========

Similarity Score: 0.355

    Artificial Intelligence is a branch of computer science that focuses
    on creating systems capable of performing tasks that normally require
    human intelligence. AI is used in healthcare, education, finance,
    transportation and many other fields.
    

Similarity Score: 0.29

    Natural Language Processing, or NLP, is a field of Artificial
    Intelligence that enables computers to understand and process human
    language. NLP is used in chatbots, translation, summarization and
    sentiment analysis.
    


In [7]:
def calculator_tool(expression):
    try:
        allowed_characters = "0123456789+-*/(). "

        if not all(char in allowed_characters for char in expression):
            return "Invalid mathematical expression."

        result = eval(expression, {"__builtins__": None}, {})

        return result

    except Exception:
        return "Unable to calculate the expression."

In [8]:
print("Calculator Test 1:")
print(calculator_tool("25 + 15"))

print("\nCalculator Test 2:")
print(calculator_tool("100 / 4"))

print("\nCalculator Test 3:")
print(calculator_tool("12 * 8"))

Calculator Test 1:
40

Calculator Test 2:
25.0

Calculator Test 3:
96


In [9]:
def research_agent(question):

    # Check whether the question looks mathematical
    math_pattern = r'^[0-9+\-*/().\s]+$'

    if re.match(math_pattern, question):

        result = calculator_tool(question)

        return {
            "type": "tool",
            "answer": str(result),
            "source": "Calculator Tool"
        }

    # Otherwise use RAG
    retrieved = retrieve_documents(question)

    context = "\n".join(
        item["document"] for item in retrieved
    )

    return {
        "type": "rag",
        "answer": context,
        "source": "Knowledge Base"
    }

In [10]:
print("========== RESEARCH AGENT TEST ==========")

result = research_agent("What is Machine Learning?")

print("\nSource:")
print(result["source"])

print("\nRetrieved Information:")
print(result["answer"])

========== RESEARCH AGENT TEST ==========

Source:
Knowledge Base

Retrieved Information:

    Deep Learning is a subset of Machine Learning that uses artificial
    neural networks with multiple layers. It is commonly used for image
    recognition, speech recognition and natural language processing.
    

    Machine Learning is a subset of Artificial Intelligence. It allows
    computers to learn patterns from data and make predictions or decisions
    without being explicitly programmed for every task.
    


In [11]:
print("\n========== TOOL TEST ==========")

result = research_agent("25 * 4")

print("\nSource:")
print(result["source"])

print("\nAnswer:")
print(result["answer"])


========== TOOL TEST ==========

Source:
Calculator Tool

Answer:
100


In [12]:
def reviewer_agent(question, research_result):

    answer = research_result["answer"]
    source = research_result["source"]

    if source == "Calculator Tool":

        final_answer = (
            f"The calculated result for '{question}' is {answer}."
        )

    else:

        # Clean up retrieved text
        cleaned_answer = " ".join(answer.split())

        final_answer = (
            f"Based on the knowledge base: {cleaned_answer}"
        )

    return {
        "question": question,
        "source": source,
        "final_answer": final_answer
    }

In [13]:
def ai_knowledge_assistant(question):

    print("========== AI KNOWLEDGE ASSISTANT ==========")

    print("\nUser Question:")
    print(question)

    print("\n--- Research Agent ---")

    research_result = research_agent(question)

    print("Source:", research_result["source"])

    print("\n--- Reviewer Agent ---")

    final_result = reviewer_agent(
        question,
        research_result
    )

    print("Final Answer:")
    print(final_result["final_answer"])

    return final_result

In [14]:
ai_knowledge_assistant(
    "What is Deep Learning?"
)

========== AI KNOWLEDGE ASSISTANT ==========

User Question:
What is Deep Learning?

--- Research Agent ---
Source: Knowledge Base

--- Reviewer Agent ---
Final Answer:
Based on the knowledge base: Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple layers. It is commonly used for image recognition, speech recognition and natural language processing. Machine Learning is a subset of Artificial Intelligence. It allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.


{'question': 'What is Deep Learning?',
 'source': 'Knowledge Base',
 'final_answer': 'Based on the knowledge base: Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple layers. It is commonly used for image recognition, speech recognition and natural language processing. Machine Learning is a subset of Artificial Intelligence. It allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.'}

In [15]:
ai_knowledge_assistant(
    "How is AI used in healthcare?"
)

========== AI KNOWLEDGE ASSISTANT ==========

User Question:
How is AI used in healthcare?

--- Research Agent ---
Source: Knowledge Base

--- Reviewer Agent ---
Final Answer:
Based on the knowledge base: Artificial Intelligence is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. AI is used in healthcare, education, finance, transportation and many other fields. AI in healthcare can assist with medical image analysis, disease risk prediction, clinical documentation and healthcare data analysis. AI systems should be carefully evaluated because incorrect predictions can affect medical decisions.


{'question': 'How is AI used in healthcare?',
 'source': 'Knowledge Base',
 'final_answer': 'Based on the knowledge base: Artificial Intelligence is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. AI is used in healthcare, education, finance, transportation and many other fields. AI in healthcare can assist with medical image analysis, disease risk prediction, clinical documentation and healthcare data analysis. AI systems should be carefully evaluated because incorrect predictions can affect medical decisions.'}

In [16]:
ai_knowledge_assistant(
    "45 * 12"
)

========== AI KNOWLEDGE ASSISTANT ==========

User Question:
45 * 12

--- Research Agent ---
Source: Calculator Tool

--- Reviewer Agent ---
Final Answer:
The calculated result for '45 * 12' is 540.


{'question': '45 * 12',
 'source': 'Calculator Tool',
 'final_answer': "The calculated result for '45 * 12' is 540."}

In [17]:
%%writefile app.py

import re
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Artificial Intelligence is a branch of computer science that focuses on creating systems capable of performing tasks that normally require human intelligence. AI is used in healthcare, education, finance and transportation.",

    "Machine Learning is a subset of Artificial Intelligence. It allows computers to learn patterns from data and make predictions or decisions without being explicitly programmed for every task.",

    "Deep Learning is a subset of Machine Learning that uses artificial neural networks with multiple layers. It is commonly used for image recognition, speech recognition and natural language processing.",

    "Retrieval-Augmented Generation, or RAG, combines information retrieval with generative AI. A RAG system retrieves relevant information from a knowledge source and uses it to generate an answer.",

    "AI in healthcare can assist with medical image analysis, disease risk prediction, clinical documentation and healthcare data analysis.",

    "Natural Language Processing, or NLP, enables computers to understand and process human language. NLP is used in chatbots, translation, summarization and sentiment analysis."
]

vectorizer = TfidfVectorizer()
document_vectors = vectorizer.fit_transform(documents)


def retrieve_documents(question, top_k=2):

    question_vector = vectorizer.transform([question])

    similarity_scores = cosine_similarity(
        question_vector,
        document_vectors
    )[0]

    top_indices = similarity_scores.argsort()[-top_k:][::-1]

    return [
        documents[index]
        for index in top_indices
    ]


def calculator_tool(expression):

    try:
        allowed_characters = "0123456789+-*/(). "

        if not all(
            char in allowed_characters
            for char in expression
        ):
            return "Invalid mathematical expression."

        return eval(
            expression,
            {"__builtins__": None},
            {}
        )

    except Exception:
        return "Unable to calculate."


def research_agent(question):

    math_pattern = r'^[0-9+\-*/().\s]+$'

    if re.match(math_pattern, question):

        return {
            "answer": str(
                calculator_tool(question)
            ),
            "source": "Calculator Tool"
        }

    retrieved = retrieve_documents(question)

    return {
        "answer": " ".join(retrieved),
        "source": "Knowledge Base"
    }


def reviewer_agent(question, research_result):

    if research_result["source"] == "Calculator Tool":

        return (
            f"The calculated result for '{question}' "
            f"is {research_result['answer']}."
        )

    return (
        "Based on the knowledge base: "
        + research_result["answer"]
    )


import streamlit as st

st.title("🤖 AI Knowledge Assistant")

st.write(
    "A simple agentic system using RAG, tools and multi-agent processing."
)

question = st.text_input(
    "Enter your question:"
)

if st.button("Get Answer"):

    if question:

        research_result = research_agent(question)

        final_answer = reviewer_agent(
            question,
            research_result
        )

        st.subheader("Final Answer")
        st.write(final_answer)

        st.subheader("Source")
        st.write(research_result["source"])

Writing app.py


In [22]:
!pkill -f "streamlit run app.py"

In [23]:
!pip install -q pyngrok

In [24]:
!streamlit run app.py &>/content/streamlit.log &

In [26]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb

Selecting previously unselected package cloudflared.
(Reading database ... 122809 files and directories currently installed.)
Preparing to unpack cloudflared-linux-amd64.deb ...
Unpacking cloudflared (2026.9.3) ...
Setting up cloudflared (2026.9.3) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [27]:
!pkill -f streamlit

In [28]:
!streamlit run app.py &>/content/streamlit.log &

In [29]:
!cloudflared tunnel --url http://localhost:8501 --no-autoupdate

2026-09-27T12:41:35Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-09-27T12:41:35Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-09-27T12:41:38Z INF +--------------------------------------------------------------------------------------------+
2026-09-27T12:41:38Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-09-27T12:41:38Z INF |  https://venues-prot-anne-polyphonic.trycloudflare.com